In [3]:
import re
from pathlib import Path
from typing import Dict, List

#code for finding naming missmatches in optuna suggest calls. We require this to be consistent for hyperparameter optimization to work properly as we later enqueue trials with specific parameter names.

def find_suggest_mismatches(code: str, filename: str) -> List[dict]:
    """Find mismatches between trial.suggest parameter names and their usage."""

    mismatches = []

    # TYPE 1: Direct inline mismatches
    # Pattern: "dict_key": trial.suggest_*("param_name", ...)
    direct_pattern = r'"(\w+)":\s*trial\.suggest_(?:float|int|categorical|loguniform|uniform|discrete_uniform)\s*\(\s*"(?!\1[",\)])(\w+)"'

    for match in re.finditer(direct_pattern, code):
        dict_key = match.group(1)
        param_name = match.group(2)
        line_num = code[:match.start()].count('\n') + 1

        mismatches.append({
            'filename': filename,
            'line': line_num,
            'type': 'direct',
            'dict_key': dict_key,
            'variable': None,
            'suggest_param': param_name,
            'expected': f'"{param_name}": trial.suggest_*("{param_name}", ...)',
            'found': f'"{dict_key}": trial.suggest_*("{param_name}", ...)'
        })

    # TYPE 2: Indirect mismatches (variable assignment then dictionary use)
    # Step 1: Find all trial.suggest_* calls and their variable assignments
    suggest_pattern = r'(\w+)\s*=\s*trial\.suggest_(?:float|int|categorical|loguniform|uniform|discrete_uniform)\s*\(\s*"(\w+)"'

    # Map: variable_name -> suggested_param_name
    var_to_param: Dict[str, str] = {}

    for match in re.finditer(suggest_pattern, code):
        var_name = match.group(1)
        param_name = match.group(2)
        var_to_param[var_name] = param_name

    # Step 2: Find dictionary assignments using these variables
    dict_pattern = r'"(\w+)":\s*(\w+)\s*[,}]'

    for match in re.finditer(dict_pattern, code):
        dict_key = match.group(1)
        var_name = match.group(2)

        # Check if this variable came from a trial.suggest_* call
        if var_name in var_to_param:
            param_name = var_to_param[var_name]

            # Check for mismatch: dict_key should match param_name
            if dict_key != param_name:
                line_num = code[:match.start()].count('\n') + 1
                mismatches.append({
                    'filename': filename,
                    'line': line_num,
                    'type': 'indirect',
                    'dict_key': dict_key,
                    'variable': var_name,
                    'suggest_param': param_name,
                    'expected': f'"{param_name}": {var_name}',
                    'found': f'"{dict_key}": {var_name}'
                })

    return mismatches

def scan_directory(root_path: Path) -> List[dict]:
    """Scan all Python files in directory and subdirectories."""
    all_mismatches = []

    # Find all .py files recursively
    python_files = list(root_path.rglob("*.py"))

    print(f"Scanning {len(python_files)} Python files in {root_path}...\n")

    for py_file in python_files:
        try:
            code = py_file.read_text(encoding='utf-8')
            mismatches = find_suggest_mismatches(code, str(py_file.relative_to(root_path)))

            if mismatches:
                all_mismatches.extend(mismatches)

        except Exception as e:
            print(f"Error reading {py_file}: {e}")

    return all_mismatches

def print_results(mismatches: List[dict]):
    """Print all mismatches in a readable format."""
    if not mismatches:
        print("✓ No mismatches found!")
        return

    # Separate by type
    direct = [m for m in mismatches if m['type'] == 'direct']
    indirect = [m for m in mismatches if m['type'] == 'indirect']

    print(f"Found {len(mismatches)} mismatch(es):")
    print(f"  - {len(direct)} direct (inline) mismatches")
    print(f"  - {len(indirect)} indirect (variable) mismatches\n")
    print("=" * 80)

    if direct:
        print("\n📍 DIRECT MISMATCHES (inline trial.suggest):")
        for i, m in enumerate(direct, 1):
            print(f"\n{i}. {m['filename']}:{m['line']}")
            print(f"   Dict key '{m['dict_key']}' but suggest param is '{m['suggest_param']}'")
            print(f"   Expected: {m['expected']}")
            print(f"   Found:    {m['found']}")

    if indirect:
        print(f"\n📍 INDIRECT MISMATCHES (variable assignment):")
        for i, m in enumerate(indirect, 1):
            print(f"\n{i}. {m['filename']}:{m['line']}")
            print(f"   Dict key '{m['dict_key']}' uses variable '{m['variable']}'")
            print(f"   But '{m['variable']}' was created with suggest param '{m['suggest_param']}'")
            print(f"   Expected: {m['expected']}")
            print(f"   Found:    {m['found']}")

    print("\n" + "=" * 80)

# Main execution
if __name__ == "__main__":
    # Get the current directory (where the notebook/script is)
    current_dir = Path.cwd()

    # Or specify a custom path:
    # current_dir = Path("/path/to/your/project")

    print(f"Searching in: {current_dir}\n")

    # Scan all files
    mismatches = scan_directory(current_dir)

    # Print results
    print_results(mismatches)

    # Optionally, save to a file
    if mismatches:
        output_file = current_dir / "optuna_mismatches.txt"
        with open(output_file, 'w') as f:
            f.write(f"Optuna Parameter Mismatches Report\n")
            f.write(f"{'=' * 80}\n\n")
            for m in mismatches:
                f.write(f"{m['filename']}:{m['line']}\n")
                f.write(f"  Expected: {m['expected']}\n")
                f.write(f"  Found:    {m['found']}\n\n")
        print(f"\nResults saved to: {output_file}")

Searching in: C:\Users\Lindner\PycharmProjects\experiments_rotation\experiment_thesis\ood

Scanning 30 Python files in C:\Users\Lindner\PycharmProjects\experiments_rotation\experiment_thesis\ood...

✓ No mismatches found!
